# EduPlus: Deep Knowledge Tracing (DKT) Model Training
### Kaggle-Ready Notebook for Training & Exporting Student Mastery Models

This notebook trains a Recurrent Neural Network (RNN / LSTM) to model student knowledge over sequences of quiz questions. Once trained, the exported model can be plugged directly into EduPlus's `backend/knowledge_tracing.py` interface (`estimate_level`).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training using device: {device}")

## 1. Synthetic / Kaggle Dataset Generator
Simulates student sequences across 4 CBSE subjects (Mathematics, Physics, Chemistry, English) with skill tags.

In [ ]:
NUM_CONCEPTS = 20  # 5 concepts per 4 CBSE subjects
SEQ_LEN = 15
NUM_STUDENTS = 2000

def generate_synthetic_data(num_students=2000, seq_len=15, num_concepts=20):
    # Each interaction is a tuple: (concept_id, is_correct)
    # Encoded as: concept_id + is_correct * num_concepts
    X = []
    Y = []
    
    for _ in range(num_students):
        student_ability = np.random.normal(0.6, 0.2)
        concepts = np.random.randint(0, num_concepts, size=seq_len)
        results = (np.random.rand(seq_len) < student_ability).astype(int)
        
        # Features: combined concept + correctness
        features = concepts + (results * num_concepts)
        X.append(features[:-1])
        Y.append(results[1:])
        
    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.float32)

train_X, train_Y = generate_synthetic_data(1600)
val_X, val_Y = generate_synthetic_data(400)
print(f"Dataset generated: {train_X.shape[0]} training, {val_X.shape[0]} validation sequences.")

## 2. Deep Knowledge Tracing (DKT) Architecture

In [ ]:
class DeepKnowledgeTracing(nn.Module):
    def __init__(self, num_inputs, hidden_dim, num_outputs):
        super(DeepKnowledgeTracing, self).__init__()
        self.embedding = nn.Embedding(num_inputs, hidden_dim)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_outputs)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        logits = self.fc(lstm_out)
        return self.sigmoid(logits)

model = DeepKnowledgeTracing(num_inputs=NUM_CONCEPTS * 2, hidden_dim=64, num_outputs=1).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)
print(model)

## 3. Training Loop

In [ ]:
EPOCHS = 10
BATCH_SIZE = 64

for epoch in range(EPOCHS):
    model.train()
    permutation = torch.randperm(train_X.size()[0])
    total_loss = 0.0
    
    for i in range(0, train_X.size()[0], BATCH_SIZE):
        indices = permutation[i:i + BATCH_SIZE]
        batch_x, batch_y = train_X[indices].to(device), train_Y[indices].unsqueeze(-1).to(device)
        
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    # Validation check
    model.eval()
    with torch.no_grad():
        val_preds = model(val_X.to(device)).squeeze().cpu().numpy().flatten()
        val_targets = val_Y.numpy().flatten()
        auc = roc_auc_score(val_targets, val_preds)
        
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(indices):.4f} | Validation AUC: {auc:.4f}")

## 4. Export Model for EduPlus Integration
Exports PyTorch model weights to ONNX format, which can be loaded by Python ONNXRuntime in `backend/knowledge_tracing.py`.

In [ ]:
dummy_input = torch.randint(0, NUM_CONCEPTS * 2, (1, 14)).to(device)
torch.onnx.export(
    model,
    dummy_input,
    "../backend/knowledge_tracing_model.onnx",
    export_params=True,
    input_names=['input_interactions'],
    output_names=['mastery_prediction'],
    dynamic_axes={'input_interactions': {0: 'batch_size', 1: 'sequence_length'}}
)
print("Model exported successfully to backend/knowledge_tracing_model.onnx!")